# Colour Brings People Together
### Part B · Auditing the UN SDG palette across languages, colour vision and age

The United Nations identifies its 17 Sustainable Development Goals with 17 colours, used unchanged in every country and every language. In the next 45 minutes you and your partner will test how well that palette communicates along two axes:

| Axis | Question | Assessed |
|---|---|---|
| **Perceptual** | Can people still tell the colours apart with a colour vision deficiency, or with an ageing eye? | within a community |
| **Cognitive** | Do people reliably agree on what each colour is called, and do the names in different languages point to the same colour? | across languages |

A palette can succeed on one axis and fail on the other. By the end you will have a two-axis picture of a subset of SDG colours and a comparison with what an AI system says the same colours are called.

**How to use this notebook.** Run each cell in order with **Shift + Enter**. The only things you need to change are the grey form fields on the right of some cells. Keep the [Colour Spinner](https://colornaming.net/spinner) open in another tab.

In [ ]:
#@title Run me first: load the masterclass toolkit
#@markdown The colour maths, the Colour Spinner connection and the charts live in `colour_masterclass.py`, published next to these notebooks.
TOOLKIT_URL = "https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/colour_masterclass.py"  #@param {type:"string"}
import os, urllib.request
if not os.path.exists("colour_masterclass.py"):
    try:
        urllib.request.urlretrieve(TOOLKIT_URL, "colour_masterclass.py")
    except Exception as e:
        raise RuntimeError("Could not download the toolkit. Check your internet connection, or upload "
                           "colour_masterclass.py in Colab's Files panel (folder icon, left).") from e
from colour_masterclass import *

## 1 · Meet the palette

Every swatch in this notebook sits on a neutral mid-grey surround. A white or black page changes how a colour looks, so colour scientists judge colours against grey.

In [ ]:
show_palette(SDG.hex.tolist(), labels=SDG.index.tolist(), captions=SDG.goal.tolist(),
             note="The 17 UN Sustainable Development Goal colours (United Nations 2015).")

## 2 · Choose your subset and languages

Pick **three to five** SDG colours. Good audits mix an easy colour with a hard one: try one of the reds (1, 4, 5, 8), one of the blues (6, 14, 16, 17) and something in between such as 12.

Tick the languages you want to audit. The Colour Spinner currently covers six. If either of you speaks one of them natively, include it, and include **Himba**, whose speakers partition colour very differently from European languages.

**Backend.** *Colour Spinner API* fetches everything for you. If the network is slow, choose *Copied from the browser* and fill in the sheet in section 4.

In [ ]:
#@title Your pair's choices { run: "auto" }
SUBSET = "1, 6, 12, 14"  #@param {type:"string"}
SOURCE_LANGUAGE = "British English"  #@param ["American English", "British English", "French", "Greek", "Himba", "Spanish"]
American_English = False  #@param {type:"boolean"}
British_English = True  #@param {type:"boolean"}
French = False  #@param {type:"boolean"}
Greek = True  #@param {type:"boolean"}
Himba = True  #@param {type:"boolean"}
Spanish = True  #@param {type:"boolean"}
BACKEND = "Colour Spinner API"  #@param ["Colour Spinner API", "Copied from the browser"]

ticked = {"American English": American_English, "British English": British_English,
          "French": French, "Greek": Greek, "Himba": Himba, "Spanish": Spanish}
LANGUAGES = pick_languages(SOURCE_LANGUAGE, ticked)
PICK = parse_subset(SUBSET)
show_subset(PICK, LANGUAGES)

## 3 · The perceptual axis: can everyone tell them apart?

About 1 in 12 men and 1 in 200 women of Northern European descent have a colour vision deficiency, most often a reduced ability to separate reds from greens. Independently, the lens of the eye yellows with age and absorbs more short-wavelength light, so blues and greens drift towards one another.

The rows below show the whole palette under each condition, simulated by the Colour Spinner's own models, the same ones you see with its "Simulate age/CVD" menu. The age rows are age bands (16–40, 40–60, 60–80). Look down each column for colours that change, and across each row for colours that merge.

In [ ]:
prefetch_simulations(SDG.hex.tolist())      # ask the Colour Spinner for all 17 colours at once
show_conditions(SDG.hex.tolist(), labels=[str(i) for i in SDG.index])

The chart below makes "merge" measurable. For each colour in your subset it finds the **nearest other SDG colour** under each condition and reports the CIEDE2000 colour difference (ΔE00) between them. A difference of about 1 is just noticeable side by side; for colours that must be told apart at a glance in a chart or a sign, aim for 10 or more. The small number under each value names the nearest neighbour.

In [ ]:
#@title Nearest-neighbour distinctiveness { run: "auto" }
DISTINCT_AT = 10  #@param {type:"slider", min:3, max:25, step:1}

nn_de, nn_id = nearest_neighbours(PICK)      # ΔE00 to the nearest other SDG colour, per condition
plot_nearest_neighbours(nn_de, nn_id, DISTINCT_AT)
show_at_risk(nn_de, nn_id, DISTINCT_AT)

## 4 · The cognitive axis: what are these colours called?

Here you ask the Colour Spinner what speakers of each language actually call your colours. Its data come from thousands of people who named colours online at [colornaming.net](https://colornaming.net). Treat it as a reference baseline rather than ground truth: it currently covers six languages and only the essential colour terms in each, the minimal set that can name every colour.

Two measures come out of this step:

- **Nameability**: the Namer's *confidence*, the share of speakers who would use the top name. Low confidence means people disagree.
- **Translatability**: whether each language's name points to the same colour. The Mapper's *best example* of each name shows where that name is centred; if the best examples drift apart across languages, a "translation" of the colour lands somewhere else.

**Only if you chose *Copied from the browser*:** fill in the sheet below, one row per colour and language. In the Spinner, type the hex into the Picker, set the Namer to the language, and copy the name, its confidence and the Mapper's best-example RGB. Otherwise skip this cell.

In [ ]:
SHEET = """
hex, language, name, confidence, best_example
# C34F4F, American English, Red, 20%, 232 21 56      ← example row, as the Spinner shows #C34F4F
"""

In [ ]:
spinner = connect(BACKEND, SHEET)

rows, problems = [], []
for sdg in PICK:
    for language in LANGUAGES:
        colour = SDG.hex[sdg]
        try:
            namer = spinner.name(colour, language)                  # Namer: top name and agreement
        except Exception as e:
            problems.append(f"SDG {sdg} in {language}: {e}"); continue
        best = safe(spinner.best_example, namer["name"], language)  # Mapper: where that name is centred
        rows.append({"sdg": sdg, "hex": colour, "language": language,
                     "name": namer["name"], "romanized": namer.get("romanized"),   # Latin spelling, e.g. kokkino
                     "confidence": namer["confidence"], "best_example": best,
                     "drift": de00(colour, best) if best else np.nan})  # how far the name pulls the colour
naming = pd.DataFrame(rows)

report_problems(problems)
show_names_table(naming, LANGUAGES)

### Nameability

Each marker is one language. Colours near the right are named consistently; colours on the left are where speakers disagree, and where a label such as "the red goal" will mean different things to different readers.

In [ ]:
plot_nameability(naming, PICK, LANGUAGES)

### Translatability

The left swatch is the SDG colour. Each swatch to its right is the **best example** of the name that language uses for it, with its distance from the original. When those swatches differ from each other, the "same" colour word does not mean the same colour in each language.

In [ ]:
show_translatability(naming, PICK, LANGUAGES, DISTINCT_AT)

The same information as a map. This is a top-down view of CAM16-UCS colour space, the perceptually uniform space the Colour Spinner works in: hue runs around the centre and colourfulness increases outwards. Each large disc is an SDG colour; the lines lead to where each language's name for it is centred.

In [ ]:
plot_name_map(naming, PICK, LANGUAGES)

## 5 · Two axes, one picture

Each disc is a colour from your subset. Its horizontal position is the **worst case** on the perceptual axis: the smallest nearest-neighbour ΔE00 across all seven viewing conditions. Its vertical position is the **average naming agreement** across your languages.

In [ ]:
#@title Scorecard { run: "auto" }
AGREE_AT = 0.5  #@param {type:"slider", min:0.2, max:0.9, step:0.05}

score = scorecard(nn_de, naming)
plot_two_axes(score, DISTINCT_AT, AGREE_AT)
display(score.round(2))

## 6 · What does an AI system say?

Now ask a current AI system to name the same colours in the same languages, and compare its answers with how speakers actually name them.

**Which AI?** We demonstrate with the Gemini model built into Colab because it needs no set-up. It stands in for AI systems in general and isn't a recommendation. You are welcome to use the AI you use day to day (ChatGPT, Claude, Copilot, …) instead, and the comparison is often more meaningful that way.

Choose in the form below:

- **Colab's Gemini:** leave the form as it is and run the cells in order.
- **Your own AI:** choose *Paste the reply from my own AI*. The next cell shows the prompt to copy into your AI tool, and a box to paste its reply into. The reply can be in any reasonable layout: lines, a table or JSON.

In [ ]:
#@title Which AI?
AI_PROVIDER = "Gemini built into Colab (no key needed)"  #@param ["Gemini built into Colab (no key needed)", "Paste the reply from my own AI", "Anthropic", "OpenAI-compatible"]
AI_MODEL = ""  #@param {type:"string"}
AI_BASE_URL = ""  #@param {type:"string"}
#@markdown **Gemini built into Colab** needs nothing more: just run the cells below.
#@markdown **Your own AI** (ChatGPT, Claude, Copilot, …): choose *Paste the reply from my own AI*; the next cell gives you the prompt and a box to paste the reply into.
#@markdown *Anthropic* and *OpenAI-compatible* call an API with your own key from Colab ▸ Secrets (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) and a model name. Leave AI_MODEL empty for Colab's default Gemini.

In [ ]:
lang_note = "; ".join(f"{l}" + (" (Otjiherero, spoken by the Himba of Namibia)" if l == "Himba" else "")
                      for l in LANGUAGES)
PROMPT_B = f"""You are helping with a cross-cultural colour naming study.
For each colour below, give the single colour name that most native speakers of each language would use for it.
Write each name in that language and in its usual script; for a non-Latin script, add the Latin spelling in brackets.

Languages: {lang_note}
Colours (sRGB hex): {", ".join(SDG.hex[i] for i in PICK)}

Reply with one line for every colour and language, and nothing else, in exactly this format:
#E5243B | {LANGUAGES[-1]} | name"""
show_prompt(PROMPT_B, AI_PROVIDER)

In [ ]:
reply_b = ai_reply(PROMPT_B, AI_PROVIDER, AI_MODEL, AI_BASE_URL)
ai_names = read_ai_names(reply_b)

comparison = naming.merge(ai_names, on=["hex", "language"], how="left")
comparison["agrees"] = [same_name(ai, name, rom)    # any script; ignores case and accents
                        for ai, name, rom in zip(comparison.ai_name, comparison.name, comparison.romanized)]
comparison = where_ai_names_point(comparison, spinner)   # Mapper: where each differing AI name is centred
show_name_comparison(comparison, DISTINCT_AT)
plot_agreement_by_language(comparison, LANGUAGES)

A match on a colour with low speaker agreement says little: when only a third of speakers use the top name, other answers are also defensible. So for every name that differs, the last column asks the Spinner's Mapper where the AI's name is centred for speakers of that language. If that colour is close to the SDG colour, the AI chose a plausible neighbouring name. If it is far away, or the name isn't one speakers use at all, the AI's name would mislead them. Large distances on high-agreement colours are the ones to bring to the group.

## 7 · Save your results for Part C

Part C picks up the names you found here. Run the next cell to download `partB_results.csv`. If you are working in Colab, keep the tab open: Part C can also reload it from your Downloads.

In [ ]:
out = naming.merge(ai_names, on=["hex", "language"], how="left") if "ai_names" in globals() else naming
out.to_csv("partB_results.csv", index=False)
download("partB_results.csv")

## Before you share with the room

Agree with your partner on one sentence for each:

1. Which colour performs best on both axes, and which worst?
2. Is your worst colour failing **perceptually**, **cross-linguistically** or **both**?
3. Where did the AI system's names diverge from how speakers name the colour, and does the divergence look like a default towards English or "universal" colour categories?